# Chapter 44: European option pricing and hedging

Yuxiao Ma

All paths are SIMULATED/CONSTRUCTED. Fresh mechanisms use shared pricing/stochastic functions, without market data or training. Time is measured in years; total-return drift mu includes dividends, so P price drift is mu−q and Q drift is r−q. The full CLI protocol is separate. This Notebook saves three new PNGs, not a complete CLI grid.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch44_stochastic_derivatives_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from quant_course.stochastic_calculus import brownian_increments, ito_left, quadratic_variation, gbm_exact, gbm_euler
from quant_course.derivatives_pricing import bs_price, bs_greeks, crr_price, mc_price, delta_hedge
p=dict(S=100.,K=100.,T=1.,r=.05,q=.02,sigma=.2)
print("SIMULATED", {k:bs_price(**p,kind=k) for k in ["call","put"]})

## Left-point Itô sum and quadratic variation

For 16 paths with 1,024 increments, verify the finite-grid identity $I=\tfrac12(W_T^2-QV)$. Only in the appropriate limit does QV become T; do not force finite path differences to zero. Save four illustrative paths in the first figure.

In [ ]:
dw=brownian_increments(16,1024,1.,4402)
w=np.column_stack([np.zeros(16),np.cumsum(dw,axis=1)])
qv=quadratic_variation(w); integ=ito_left(w,w[:,:-1])
assert np.allclose(integ,.5*(w[:,-1]**2-qv),atol=1e-12)
print("finite identity max residual",max(abs(integ-.5*(w[:,-1]**2-qv))))
fig,ax=plt.subplots(figsize=(8,3)); ax.plot(np.linspace(0,1,1025),w[:4].T)
ax.set(xlabel="Time (years)",ylabel="W (sqrt(years))",title="SIMULATED Brownian paths")
plt.savefig(RUN_DIR / "ch44_stochastic_derivatives_cell03.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close(); plt.close(fig)

## Common-noise GBM and Euler

Generate 128 paths at 1,024 steps, aggregate the same noise to 16/64/256/1,024 steps and compare terminal exact/Euler values. Report strong RMSE and paired weak-error mean/SE. Independent noise would confound discretization error with sampling variation. This is smaller than the CLI's 4,096-path study.

In [ ]:
dw2=brownian_increments(128,1024,1.,4403)
rows=[]
for steps in [16,64,256,1024]:
    d=dw2.reshape(128,steps,1024//steps).sum(axis=2)
    ex=gbm_exact(100.,.06,.2,1/steps,d)
    eu=gbm_euler(100.,.06,.2,1/steps,d)
    err=eu[:,-1]-ex[:,-1]
    rows.append((steps,np.sqrt(np.mean(err**2)),np.mean(err),np.std(err,ddof=1)/np.sqrt(128)))
print("steps,strong RMSE,paired weak mean,paired SE",rows)
fig,ax=plt.subplots(); ax.loglog([x[0] for x in rows],[x[1] for x in rows],"o-")
ax.set(xlabel="Steps",ylabel="Terminal strong RMSE (currency/share)",title="SIMULATED common noise")
plt.savefig(RUN_DIR / "ch44_stochastic_derivatives_cell05.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close(); plt.close(fig)

## CRR, Black–Scholes and Q Monte Carlo

Compare eight tree depths, including odd/even pairs, and 5,000 antithetic pairs. SE uses independent pair averages; 95% intervals are approximate normal intervals. A single covered benchmark does not validate every implementation. Save the tree-convergence figure.

In [ ]:
bs=bs_price(**p)
tree=[(s,crr_price(**p,steps=s)-bs) for s in [10,25,50,51,100,101,200,400]]
m=mc_price(**p,pairs=5000,seed=4401,antithetic=True)
print("CRR minus BS",tree); print("Q MC",m)
assert abs(bs_price(**p)-bs_price(**p,kind="put")-100*np.exp(-.02)+100*np.exp(-.05))<1e-10
fig,ax=plt.subplots(); ax.plot(*zip(*tree),"o-"); ax.axhline(0,color="gray")
ax.set(xlabel="Tree steps",ylabel="CRR minus BS (currency/share)",title="CONSTRUCTED tree convergence")
plt.savefig(RUN_DIR / "ch44_stochastic_derivatives_cell07.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close(); plt.close(fig)

## Greek units

For spots 80, 100 and 120 and both call and put, compare Vega with central finite differences using a step of 10⁻⁴. Vega is per unit volatility; multiply by 0.01 for one percentage point. Theta is per calendar year. The API excludes T=0 and sigma=0 Greeks. This defines its implemented domain rather than asserting that every limiting derivative is undefined.

In [ ]:
for spot in [80.,100.,120.]:
    for kind in ["call","put"]:
        a=dict(p,S=spot); g=bs_greeks(**a,kind=kind)
        h=1e-4
        fd=(bs_price(**dict(a,sigma=.2+h),kind=kind)-bs_price(**dict(a,sigma=.2-h),kind=kind))/(2*h)
        assert abs(fd-g["vega"])<1e-4
        print(spot,kind,g,"vega 1pp",.01*g["vega"])


## Discrete P-measure delta hedge

Directly generate one 12-step path with seed 4404 and P price drift 0.06. Compare transaction-cost rates of zero and 0.001, including initial and terminal trades. Left-point shares earn interval returns; dividend cash uses a left-point approximation. The CLI first draws a larger fine-noise array and aggregates it, so the same seed does not identify the same saved path.

In [ ]:
dw3=brownian_increments(1,12,1.,4404)
path=gbm_exact(100.,.06,.2,1/12,dw3)[0]
for fee in [0.,.001]:
    hedge=delta_hedge(path,1.,.05,.02,.2,100.,cost_rate=fee)
    print("fee",fee,"portfolio minus payoff",hedge["error"],"cash cost",hedge["total_cost"])
    print("first and final ledger",hedge["ledger"][0],hedge["ledger"][-1])

## Information counterexamples

Changing increments after the prefix does not alter its left-point sum. Array shape checks do not prove adaptedness: the anticipating W_T integrand uses future information even though it passes shape validation. The caller must establish the information boundary.

In [ ]:
future=w.copy(); future[:,513:]+=3.
past1=ito_left(w[:,:513],w[:,:512]); past2=ito_left(future[:,:513],future[:,:512])
assert np.array_equal(past1,past2)
# Anticipating integrand W_T uses future information: E[W_T*W_T]=T, not zero.
anticipating=ito_left(w,np.repeat(w[:,-1,None],1024,axis=1))
assert np.allclose(anticipating,w[:,-1]**2)
print("prefix invariant",True,"anticipating path0",anticipating[0])